In [1]:
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="Qwen/Qwen3-8B", base_url="https://api.siliconflow.cn")

from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver


def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"


agent = create_agent(
    model,
    tools=[get_weather]
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

##updates

In [2]:
for chunk in agent.stream(  
    {"messages": [{"role": "user", "content": "杭州天气如何"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'id': '019b43b2ff83bcbd5c9bb3a5ccebb121', 'name': 'get_weather', 'args': {'city': '杭州'}}]
step: tools
content: [{'type': 'text', 'text': "It's always sunny in 杭州!"}]
step: model
content: [{'type': 'text', 'text': '杭州的天气总是阳光明媚！需要我为你提供更详细的天气信息吗？比如温度、湿度或者风力情况？'}]


##messages 记录token消耗

In [3]:
for token, metadata in agent.stream(  
    {"messages": [{"role": "user", "content": "杭州天气如何"}]},
    stream_mode="messages",
):
    print(f"node: {metadata['langgraph_node']}")
    print(f"content: {token.content_blocks}")
    print(f"token: {token}")
    print("\n")

node: model
content: []
token: content='' additional_kwargs={} response_metadata={} id='lc_run--28cf6f34-d57e-40bd-b32a-d5bc6ae8f5b9' usage_metadata={'input_tokens': 157, 'output_tokens': 0, 'total_tokens': 157, 'input_token_details': {}, 'output_token_details': {}}


node: model
content: [{'type': 'tool_call_chunk', 'id': '019b43b309cf6e6764a695a1890a54ac', 'name': 'get_weather', 'args': '', 'index': 0}]
token: content='' additional_kwargs={'tool_calls': [{'index': 0, 'id': '019b43b309cf6e6764a695a1890a54ac', 'function': {'arguments': '', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={} id='lc_run--28cf6f34-d57e-40bd-b32a-d5bc6ae8f5b9' tool_calls=[{'name': 'get_weather', 'args': {}, 'id': '019b43b309cf6e6764a695a1890a54ac', 'type': 'tool_call'}] usage_metadata={'input_tokens': 157, 'output_tokens': 9, 'total_tokens': 166, 'input_token_details': {}, 'output_token_details': {}} tool_call_chunks=[{'name': 'get_weather', 'args': '', 'id': '019b43b309cf6e6764a695a1890a54a

In [4]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer  


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()  
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

agent = create_agent(
    model,
    tools=[get_weather]
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "杭州天气怎么样"}]},
    stream_mode="custom"
):
    print(chunk)

Looking up data for city: 杭州
Acquired data for city: 杭州
